# Step 03 of 04 - Copy-move manipulation generator

To teach the segmentation model what a manipulation is we implemented a copy-move generator.
A local patch around one digit is copied and pasted over another digit. Each
sample is paired with a binary mask marking the tampered pixels.

For the train set 5 times more *copy-move* samples for every *clean* sample, to let the model learn manipulation and then it was gradually amended in the validation and test set to understand how it would perform on ratios closer to the real world.

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

## Configuration & data structures

All tunable parameters

- **SplitGenerationConfig** - how many clean vs. copy-move
  samples to generate per crop, per split.
- **Digit-detection params** - area/aspect bounds used to find digit blobs.
- **Copy-move realism params** - jitter, noise, JPEG recompression, patch
  feathering, and the no-op rejection thresholds that discard manipulations too
  subtle to be meaningful.
- **DigitBox** - a simple x/y/w/h bounding box.

In [ ]:
# Notebook runs from backend/ml/notebooks, so the ml dir is the parent of cwd
ML_DIR = Path.cwd().parent

# === Config ===
INPUT_ROOT  = ML_DIR / "data/stage_02/date_crops"
OUTPUT_ROOT = ML_DIR / "data/stage_02/dataset"
RANDOM_SEED = 42

# Copy-move and data augmentation generator
# This is used specifically to expand the date field crops dataset
# It is NOT yet tested on other field crops
@dataclass(frozen=True)
class SplitGenerationConfig:
    clean_per_crop: int
    copy_move_per_crop: int
    max_attempt_multiplier: int = 10


SPLIT_CONFIGS: Dict[str, SplitGenerationConfig] = {
    'train': SplitGenerationConfig(clean_per_crop=30, copy_move_per_crop=150),
    'val':   SplitGenerationConfig(clean_per_crop=50, copy_move_per_crop=20),
    'test':  SplitGenerationConfig(clean_per_crop=90, copy_move_per_crop=10),
}

CLS_CLEAN = 0
CLS_COPY_MOVE = 1

# ============================================================
# Digit detection params
# ============================================================
DIGIT_MIN_AREA = 100
DIGIT_MAX_AREA = 3000
DIGIT_MIN_ASPECT = 0.1
DIGIT_MAX_ASPECT = 2.0
DATE_REGION_X_START_RATIO = 0.30

# ============================================================
# Copy-move realism
# ============================================================
POSITION_JITTER_PX = 2
GAUSSIAN_NOISE_SIGMA = (0.0, 1.5)
JPEG_PROB = 0.70
JPEG_QUALITY_RANGE = (50, 100)

# Source/target matching
SIZE_MATCH_RATIO = 0.80
SIZE_TOLERANCE = 0.25
PAIR_SELECTION_ATTEMPTS = 80

# Reject likely no-op manipulations.
SAME_SHAPE_IOU_THRESHOLD = 0.85
MIN_TAMPERED_PIXELS = 20
MIN_MEAN_ABS_DIFF_IN_MASK = 3.0

SOURCE_MARGIN_PX_RANGE = (2, 6)
PATCH_SCALE_JITTER = (0.95, 1.05)       # tiny resize jitter
PATCH_ROTATION_DEGREES = (-1.0, 1.0)    # tiny rotation jitter

# Patch alpha: keep patch interior opaque, only feather the outer border.
MANIPULATION_STYLES = {
    'patch_hard':      0.35,
    'patch_feathered': 0.65,
}
FEATHER_SIGMA_OPTIONS = [0.4, 0.7, 1.0, 1.3]
BORDER_FEATHER_MIN_ALPHA = 0.85
MASK_ALPHA_THRESHOLD = 0.50

# Source-patch edits. Keep subtle, to not let model learn fake cues.
COLOR_TINT_PROB = 0.12
BRIGHTNESS_OFFSET_PROB = 0.18
BRIGHTNESS_OFFSET_RANGE = (-8, 8)


@dataclass(frozen=True)
class DigitBox:
    x: int
    y: int
    w: int
    h: int

## Utility helpers

Small building blocks: listing image paths, restricting digits to the right-hand
date region, checking whether two boxes are size-compatible, and cropping /
expanding boxes.

In [ ]:
# ============================================================
# Utility helpers
# ============================================================
def list_image_paths(folder: Path) -> List[Path]:
    if not folder.exists():
        raise FileNotFoundError(f'Input split folder does not exist: {folder}')
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})


def filter_digits_to_date_region(digits: List[DigitBox], crop_width: int) -> List[DigitBox]:
    x_threshold = crop_width * DATE_REGION_X_START_RATIO
    return [d for d in digits if d.x >= x_threshold]


def boxes_size_compatible(a: DigitBox, b: DigitBox) -> bool:
    w_ratio = abs(a.w - b.w) / max(a.w, b.w)
    h_ratio = abs(a.h - b.h) / max(a.h, b.h)
    return w_ratio <= SIZE_TOLERANCE and h_ratio <= SIZE_TOLERANCE


def crop_box(img: np.ndarray, box: DigitBox) -> np.ndarray:
    return img[box.y:box.y + box.h, box.x:box.x + box.w]


def expand_box(box: DigitBox, image_shape: Tuple[int, int, int], margin: int) -> DigitBox:
    h_img, w_img = image_shape[:2]
    x1 = max(0, box.x - margin)
    y1 = max(0, box.y - margin)
    x2 = min(w_img, box.x + box.w + margin)
    y2 = min(h_img, box.y + box.h + margin)
    return DigitBox(x=x1, y=y1, w=max(1, x2 - x1), h=max(1, y2 - y1))

## Digit detection & masks

`detect_digits` finds candidate digit boxes via adaptive thresholding and
connected components. `get_digit_shape_mask` extracts a per-digit shape mask used
only to reject near-identical source/target pairs.

In [ ]:
# ============================================================
# Digit detection and masks
# ============================================================
def detect_digits(crop: np.ndarray) -> List[DigitBox]:
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 25, 10
    )

    kernel = np.ones((2, 2), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    n, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    digits: List[DigitBox] = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if not (DIGIT_MIN_AREA <= area <= DIGIT_MAX_AREA):
            continue
        aspect = w / max(h, 1)
        if not (DIGIT_MIN_ASPECT <= aspect <= DIGIT_MAX_ASPECT):
            continue
        digits.append(DigitBox(int(x), int(y), int(w), int(h)))

    return sorted(digits, key=lambda d: d.x)


def get_digit_shape_mask(crop: np.ndarray, digit_box: DigitBox) -> np.ndarray:
    """Only used for no-op rejection, not for the paste alpha."""
    region = crop_box(crop, digit_box)
    if region.size == 0:
        return np.zeros((digit_box.h, digit_box.w), dtype=np.float32)

    gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((2, 2), np.uint8)
    binary = cv2.dilate(binary, kernel, iterations=1)
    mask = (binary > 0).astype(np.float32)

    if mask.shape != (digit_box.h, digit_box.w):
        mask = cv2.resize(mask, (digit_box.w, digit_box.h), interpolation=cv2.INTER_NEAREST)
    return mask

## Pair selection & no-op rejection

Picks a source and target digit to copy between. `masks_iou` and
`digit_shapes_too_similar` reject pairs whose shapes overlap too much (which would
produce an invisible forgery); `select_source_target_pair` samples size-matched
pairs preferentially.

In [ ]:
# ============================================================
# Pair rejection / no-op detection
# ============================================================
def masks_iou(a: np.ndarray, b: np.ndarray, eps: float = 1e-7) -> float:
    a_bin = a > 0.5
    b_bin = b > 0.5
    inter = np.logical_and(a_bin, b_bin).sum()
    union = np.logical_or(a_bin, b_bin).sum()
    return float(inter / (union + eps))


def digit_shapes_too_similar(crop: np.ndarray, src: DigitBox, tgt: DigitBox) -> bool:
    src_mask = get_digit_shape_mask(crop, src)
    tgt_mask = get_digit_shape_mask(crop, tgt)
    if src_mask.size == 0 or tgt_mask.size == 0:
        return True

    src_resized = cv2.resize(src_mask, (tgt_mask.shape[1], tgt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
    return masks_iou(src_resized, tgt_mask) >= SAME_SHAPE_IOU_THRESHOLD


def select_source_target_pair(digits: List[DigitBox], crop: np.ndarray) -> Tuple[Optional[DigitBox], Optional[DigitBox]]:
    if len(digits) < 2:
        return None, None

    all_pairs = [(a, b) for i, a in enumerate(digits) for j, b in enumerate(digits) if i != j]
    size_pairs = [(a, b) for a, b in all_pairs if boxes_size_compatible(a, b)]

    for _ in range(PAIR_SELECTION_ATTEMPTS):
        candidates = size_pairs if (random.random() < SIZE_MATCH_RATIO and size_pairs) else all_pairs
        if not candidates:
            return None, None
        src, tgt = random.choice(candidates)
        if digit_shapes_too_similar(crop, src, tgt):
            continue
        return src, tgt

    return None, None

## Image augmentation & post-processing

Subtle edits that make pastes look realistic and that simulate camera/codec
artifacts: color tint, brightness offset, Gaussian noise, and JPEG recompression.
`select_manipulation_style` weights hard vs. feathered patch styles.

In [ ]:
# ============================================================
# Image augmentation/post-processing
# ============================================================
def apply_color_tint(patch: np.ndarray) -> np.ndarray:
    tint = np.array([random.uniform(-4, 4), random.uniform(-4, 4), random.uniform(-4, 4)], dtype=np.float32)
    return np.clip(patch.astype(np.float32) + tint, 0, 255).astype(np.uint8)


def apply_brightness_offset(patch: np.ndarray) -> np.ndarray:
    offset = random.uniform(*BRIGHTNESS_OFFSET_RANGE)
    return np.clip(patch.astype(np.float32) + offset, 0, 255).astype(np.uint8)


def jpeg_recompress(img: np.ndarray, quality: int) -> np.ndarray:
    success, enc = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not success:
        return img
    decoded = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return decoded if decoded is not None else img


def add_gaussian_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    if sigma <= 0:
        return img
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def post_process(img: np.ndarray) -> np.ndarray:
    out = add_gaussian_noise(img, random.uniform(*GAUSSIAN_NOISE_SIGMA))
    if random.random() < JPEG_PROB:
        out = jpeg_recompress(out, random.randint(*JPEG_QUALITY_RANGE))
    return out


def select_manipulation_style() -> str:
    styles = list(MANIPULATION_STYLES.keys())
    weights = list(MANIPULATION_STYLES.values())
    return random.choices(styles, weights=weights, k=1)[0]

## Patch alpha & copy-move generation

Core forgery logic.

- **`make_patch_alpha`** - builds an alpha mask that keeps the patch interior
  opaque and only feathers the outer border.
- **`transform_patch`** - applies tiny scale/rotation jitter to the copied patch.
- **`apply_copy_move`** - copies a patch around the source digit (including
  background), pastes it over the target with jitter and blending, and returns the
  manipulated crop plus its tamper mask, rejecting near-no-op results.

In [ ]:
# ============================================================
# Patch alpha and copy-move generation
# ============================================================
def make_patch_alpha(h: int, w: int, feathered: bool, sigma: float = 1.0) -> np.ndarray:
    """Return an alpha mask for a pasted patch.

    The interior is fully opaque. Feathering affects only the outer border.
    This prevents the old target digit from showing through the pasted patch.
    """
    alpha = np.ones((h, w), dtype=np.float32)
    if not feathered or h < 3 or w < 3:
        return alpha

    # Create a rectangle surrounded by zeros, blur it, then crop the center.
    pad = max(2, int(round(3 * sigma)))
    canvas = np.zeros((h + 2 * pad, w + 2 * pad), dtype=np.float32)
    canvas[pad:pad + h, pad:pad + w] = 1.0
    soft = cv2.GaussianBlur(canvas, (0, 0), sigmaX=sigma, sigmaY=sigma)
    alpha = soft[pad:pad + h, pad:pad + w]
    alpha = np.clip(alpha, 0.0, 1.0)

    # Keep the inner area 100% opaque.
    border = max(1, int(round(2 * sigma)))
    if h > 2 * border and w > 2 * border:
        alpha[border:h - border, border:w - border] = 1.0

    # Even the feathered edge should strongly dominate the target.
    alpha[alpha > 0.05] = np.maximum(alpha[alpha > 0.05], BORDER_FEATHER_MIN_ALPHA)
    return np.clip(alpha, 0.0, 1.0).astype(np.float32)


def transform_patch(patch: np.ndarray) -> np.ndarray:
    """Tiny geometric jitter. Kept subtle because receipt text is small."""
    out = patch

    scale = random.uniform(*PATCH_SCALE_JITTER)
    if abs(scale - 1.0) > 0.01:
        new_w = max(1, int(round(out.shape[1] * scale)))
        new_h = max(1, int(round(out.shape[0] * scale)))
        out = cv2.resize(out, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    angle = random.uniform(*PATCH_ROTATION_DEGREES)
    if abs(angle) > 0.1 and out.shape[0] > 2 and out.shape[1] > 2:
        center = (out.shape[1] / 2.0, out.shape[0] / 2.0)
        mat = cv2.getRotationMatrix2D(center, angle, 1.0)
        out = cv2.warpAffine(out, mat, (out.shape[1], out.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    return out


def apply_copy_move(crop: np.ndarray, digits: List[DigitBox]) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Create one realistic local copy-move manipulation.

    This copies a source PATCH, including background around the digit, and pastes it
    over the target. It does not erase the whole image. The patch interior is opaque;
    only the patch border can be feathered.
    """
    if len(digits) < 2:
        return None, None

    style = select_manipulation_style()
    src, tgt = select_source_target_pair(digits, crop)
    if src is None or tgt is None:
        return None, None

    margin = random.randint(*SOURCE_MARGIN_PX_RANGE)
    src_patch_box = expand_box(src, crop.shape, margin)
    src_patch = crop_box(crop, src_patch_box).copy()
    if src_patch.size == 0:
        return None, None

    src_patch = transform_patch(src_patch)

    if random.random() < COLOR_TINT_PROB:
        src_patch = apply_color_tint(src_patch)
    if random.random() < BRIGHTNESS_OFFSET_PROB:
        src_patch = apply_brightness_offset(src_patch)

    patch_h, patch_w = src_patch.shape[:2]

    # Paste patch centered on the target digit, with tiny jitter.
    tgt_cx = tgt.x + tgt.w // 2
    tgt_cy = tgt.y + tgt.h // 2
    paste_x = tgt_cx - patch_w // 2 + random.randint(-POSITION_JITTER_PX, POSITION_JITTER_PX)
    paste_y = tgt_cy - patch_h // 2 + random.randint(-POSITION_JITTER_PX, POSITION_JITTER_PX)

    paste_x = max(0, min(crop.shape[1] - patch_w, paste_x))
    paste_y = max(0, min(crop.shape[0] - patch_h, paste_y))

    h_avail = min(patch_h, crop.shape[0] - paste_y)
    w_avail = min(patch_w, crop.shape[1] - paste_x)
    if h_avail <= 0 or w_avail <= 0:
        return None, None

    src_patch = src_patch[:h_avail, :w_avail]
    feathered = style == 'patch_feathered'
    sigma = random.choice(FEATHER_SIGMA_OPTIONS) if feathered else 0.0
    alpha = make_patch_alpha(h_avail, w_avail, feathered=feathered, sigma=sigma)

    manipulated = crop.copy()
    region = manipulated[paste_y:paste_y + h_avail, paste_x:paste_x + w_avail].astype(np.float32)
    src_f = src_patch.astype(np.float32)
    blended = src_f * alpha[..., None] + region * (1.0 - alpha[..., None])
    manipulated[paste_y:paste_y + h_avail, paste_x:paste_x + w_avail] = np.clip(blended, 0, 255).astype(np.uint8)

    full_mask = np.zeros(crop.shape[:2], dtype=np.uint8)
    full_mask[paste_y:paste_y + h_avail, paste_x:paste_x + w_avail] = (alpha > MASK_ALPHA_THRESHOLD).astype(np.uint8)

    if full_mask.sum() < MIN_TAMPERED_PIXELS:
        return None, None

    # Final no-op guard: skip barely changed outputs.
    changed = np.abs(manipulated.astype(np.float32) - crop.astype(np.float32)).mean(axis=2)
    mean_diff = float(changed[full_mask > 0].mean()) if full_mask.sum() > 0 else 0.0
    if mean_diff < MIN_MEAN_ABS_DIFF_IN_MASK:
        return None, None

    return manipulated, full_mask * CLS_COPY_MOVE

## Dataset generation

Writes the output dataset.

- **`write_clean_versions`** - saves post-processed clean crops with all-zero masks.
- **`write_copy_move_versions`** - repeatedly attempts copy-move forgeries until the
  target count is reached or attempts run out.
- **`process_crop`** - detects digits in one crop and produces its clean + forged samples.
- **`main`** - seeds RNGs and runs every split, reporting clean/manipulated counts.

In [ ]:
# ============================================================
# Dataset generation
# ============================================================
def write_clean_versions(crop: np.ndarray, stem: str, out_img_dir: Path, out_mask_dir: Path, n_clean: int) -> int:
    written = 0
    for i in range(n_clean):
        clean_post = post_process(crop)
        out_name = f'{stem}_clean_{i:03d}.png'
        cv2.imwrite(str(out_img_dir / out_name), clean_post)
        cv2.imwrite(str(out_mask_dir / out_name), np.zeros(crop.shape[:2], dtype=np.uint8))
        written += 1
    return written


def write_copy_move_versions(
    crop: np.ndarray,
    digits: List[DigitBox],
    stem: str,
    out_img_dir: Path,
    out_mask_dir: Path,
    n_copy_move: int,
    max_attempt_multiplier: int,
) -> int:
    generated = 0
    attempts = 0
    max_attempts = max(1, n_copy_move * max_attempt_multiplier)

    while generated < n_copy_move and attempts < max_attempts:
        attempts += 1
        manipulated, mask = apply_copy_move(crop, digits)
        if manipulated is None or mask is None:
            continue

        manipulated_final = post_process(manipulated)
        out_name = f'{stem}_cm_{generated:03d}.png'
        cv2.imwrite(str(out_img_dir / out_name), manipulated_final)
        cv2.imwrite(str(out_mask_dir / out_name), mask.astype(np.uint8))
        generated += 1

    return generated


def process_crop(crop_path: Path, config: SplitGenerationConfig, out_img_dir: Path, out_mask_dir: Path) -> Tuple[int, int]:
    crop = cv2.imread(str(crop_path))
    if crop is None:
        print(f'Could not read image: {crop_path}')
        return 0, 0

    stem = crop_path.stem
    n_clean = write_clean_versions(crop, stem, out_img_dir, out_mask_dir, config.clean_per_crop)

    digits = detect_digits(crop)
    digits = filter_digits_to_date_region(digits, crop.shape[1])

    if len(digits) < 2:
        print(f'{crop_path.name}: only {len(digits)} usable digits detected; wrote clean only')
        return n_clean, 0

    n_manip = write_copy_move_versions(
        crop=crop,
        digits=digits,
        stem=stem,
        out_img_dir=out_img_dir,
        out_mask_dir=out_mask_dir,
        n_copy_move=config.copy_move_per_crop,
        max_attempt_multiplier=config.max_attempt_multiplier,
    )

    if n_manip < config.copy_move_per_crop:
        print(
            f'{crop_path.name}: generated {n_manip}/{config.copy_move_per_crop} '
            f'copy-move samples after no-op/quality filtering'
        )

    return n_clean, n_manip


def main() -> None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    for split, config in SPLIT_CONFIGS.items():
        crops_dir = INPUT_ROOT / split
        out_img_dir = OUTPUT_ROOT / 'images' / split
        out_mask_dir = OUTPUT_ROOT / 'masks' / split
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_mask_dir.mkdir(parents=True, exist_ok=True)

        image_paths = list_image_paths(crops_dir)
        total_clean = 0
        total_manip = 0

        print(
            f'\n{split}: {len(image_paths)} input crops | '
            f'{config.clean_per_crop} clean/crop + {config.copy_move_per_crop} copy-move/crop'
        )

        for crop_path in image_paths:
            n_clean, n_manip = process_crop(crop_path, config, out_img_dir, out_mask_dir)
            total_clean += n_clean
            total_manip += n_manip

        total = total_clean + total_manip
        ratio = (total_manip / total_clean) if total_clean else float('inf')
        print(
            f'{split}: {total_clean} clean + {total_manip} manipulated = {total} total '
            f'| manipulated/clean={ratio:.2f}'
        )

    print(f'\nDataset written to: {OUTPUT_ROOT}')

## Run

Generate the full dataset.

In [ ]:
main()